# 🧠 Inferring Insights from Text

In this lesson, we’ll use LLMs to infer structured insights from unstructured text.

We'll perform tasks like:
- Sentiment classification
- Emotion recognition
- Topic detection
- Entity extraction

🔍 This is particularly useful for:
- Customer complaint triage
- Risk or fraud alert pipelines
- Document classification
- Compliance analysis


In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# Retrieve the API key
api_key = os.getenv("OPENAI_API_KEY")

# Sanity check (should print a masked version)
if api_key:
    print("✅ API key loaded successfully.")
else:
    print("❌ API key not found. Please check your .env file.")

✅ API key loaded successfully.


In [2]:
# Initialize OpenAI client
client = OpenAI(api_key=api_key)


In [3]:
def get_completion(prompt, model="gpt-4o-mini"):
    response = client.responses.create(
        model=model,
        input=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.output_text

## 📨 Sample Complaint: Financial Services Context

We’ll work with this customer complaint email throughout the notebook.

It represents a frustrated customer who may be experiencing a serious issue — our job is to infer insights such as sentiment, topic, urgency, and more.


In [5]:
customer_email = """
Dear Team,

I’ve noticed unauthorized transactions on my account over the past week. 
Despite raising a ticket 3 days ago, I haven’t received any updates. 
This is extremely frustrating and concerning given the nature of the issue.

Please escalate this matter urgently — my confidence in your service is quickly eroding.

Regards,  
Sanjay Nair
"""


## 🔍 Step 1: Sentiment Classification

We’ll ask the model to infer the **overall sentiment** of the customer complaint.

This is useful for:
- Triage prioritization
- Agent tone matching
- Customer satisfaction analysis


In [6]:
prompt = f"""
Classify the sentiment of the following customer message as one of: 
"positive", "neutral", or "negative".

Message:
\"\"\"{customer_email}\"\"\"
"""

print(prompt)


Classify the sentiment of the following customer message as one of: 
"positive", "neutral", or "negative".

Message:
"""
Dear Team,

I’ve noticed unauthorized transactions on my account over the past week. 
Despite raising a ticket 3 days ago, I haven’t received any updates. 
This is extremely frustrating and concerning given the nature of the issue.

Please escalate this matter urgently — my confidence in your service is quickly eroding.

Regards,  
Sanjay Nair
"""



In [7]:
sentiment = get_completion(prompt)
print("📊 Sentiment:", sentiment)


📊 Sentiment: The sentiment of the message is "negative."


## 😡 Step 2: Emotion Detection

We’ll now extract specific emotions the customer might be expressing, such as:
- frustration
- confusion
- urgency
- anger
- disappointment
- relief

Understanding emotion is crucial in regulated domains, where tone can indicate risk, dissatisfaction, or escalation triggers.


In [8]:
prompt = f"""
Identify the emotions expressed in the following customer email. 
List them as comma-separated values.

Message:
\"\"\"{customer_email}\"\"\"
"""

emotions = get_completion(prompt)
print("🧠 Emotions Detected:", emotions)


🧠 Emotions Detected: frustration, concern, urgency, disappointment, loss of confidence


## 🏷️ Step 3: Topic Classification

We’ll ask the model to identify the **main topic** of the customer complaint.

This is useful for:
- Routing issues to the right team
- Monitoring common themes (e.g., fraud, login issues, billing)
- Structured reporting and triage


In [9]:
prompt = f"""
Classify the main topic of the following customer complaint. 
Use a short label such as: "fraud", "technical issue", "billing", "login/access", or "other".

Message:
\"\"\"{customer_email}\"\"\"
"""

topic = get_completion(prompt)
print("🏷️ Topic:", topic)


🏷️ Topic: fraud


## 📦 Step 4: Multi-Attribute Inference (Structured Output)

Let’s extract several structured insights from the same message:
- sentiment
- emotions (as a list)
- topic
- escalation_needed (boolean)
- urgency_level (low, medium, high)

We'll ask the LLM to return the result as a **valid JSON object**.


In [10]:
prompt = f"""
Extract structured information from the following customer email.

Return a JSON object with the following keys:
- sentiment: "positive", "neutral", or "negative"
- emotions: list of strings
- topic: short label like "fraud", "billing", "access", etc.
- escalation_needed: true or false
- urgency_level: "low", "medium", or "high"

Email:
\"\"\"{customer_email}\"\"\"
"""

structured_output = get_completion(prompt)
print(structured_output)


```json
{
  "sentiment": "negative",
  "emotions": ["frustration", "concern", "disappointment"],
  "topic": "fraud",
  "escalation_needed": true,
  "urgency_level": "high"
}
```


## 🧪 Step 5: Parse and Inspect Structured Output

Once we receive a JSON object from the LLM, we can:
- Load it into Python
- Access each field
- Use the values in alerts, routing rules, or databases


In [24]:
json_data = '''
{
  "sentiment": "negative",
  "emotions": ["frustration", "concern", "disappointment"],
  "topic": "fraud",
  "escalation_needed": true,
  "urgency_level": "high"
}
'''


data_dict = json.loads(json_data)

print(data_dict)
data_dict['urgency_level']

{'sentiment': 'negative', 'emotions': ['frustration', 'concern', 'disappointment'], 'topic': 'fraud', 'escalation_needed': True, 'urgency_level': 'high'}


'high'

In [ ]:
# import json

# # Convert raw string to dictionary
# try:
#     parsed = json.loads(structured_output)

#     print("✅ Parsed JSON:")
#     for k, v in parsed.items():
#         print(f"{k}: {v}")
# except json.JSONDecodeError as e:
#     print("❌ JSON Parsing Failed:", e)
#     print("\nRaw output:\n", structured_output)


❌ JSON Parsing Failed: Expecting value: line 1 column 1 (char 0)

Raw output:
 ```json
{
  "sentiment": "negative",
  "emotions": ["frustration", "concern", "disappointment"],
  "topic": "fraud",
  "escalation_needed": true,
  "urgency_level": "high"
}
```


## 🧩 Real-World Use Case: Automated Triage & Risk Detection

Teams often deal with large volumes of customer emails, audit notes, compliance feedback, and internal reports — all in unstructured text form.

### ⚠️ The Challenge:
- Manually reviewing and tagging messages is time-consuming and inconsistent.
- Urgent issues (e.g., fraud, data breaches) may not be detected fast enough.
- Reporting dashboards lack structured fields like topic, urgency, or sentiment.

### 💡 The LLM Solution:
By using large language models (LLMs), we can infer key insights from each message:
- **Topic**: What is the message about? (e.g., fraud, billing, login issue)
- **Sentiment**: Is the tone negative or neutral?
- **Emotions**: Is the user frustrated, confused, angry?
- **Urgency & Escalation**: Should this be prioritized or escalated?

LLMs return these insights in **structured JSON format**, enabling:
- 📨 **Auto-routing** to the correct team
- ⏱️ **SLA prioritization** for urgent issues
- 🧠 **Trend analysis** across thousands of messages

### ✅ Outcomes:
- Faster resolution times
- Improved customer satisfaction
- Reduced compliance risk
- Better visibility into operational pain points

This approach applies across domains:
- 📄 Regulatory reports
- 🛠️ Helpdesk tickets
- 💬 Feedback surveys
- 🧾 Financial audit logs

